# Multi-Feature Stock Price Prediction with PyTorch LSTM
This notebook trains a deep multi-layer LSTM model for **live stock price prediction**, compares different feature sets on comprehensive graphs, and demonstrates live external data ingestion for next-candle forecasting.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from lstm_model import StockLSTM
from data_pipeline import fetch_stock_data, add_technical_indicators, prepare_datasets, StockDataset
from train_and_compare import run_feature_comparison_experiments, plot_feature_correlations, plot_market_features
from live_inference import LiveStockPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device.upper()}")

## 1. Fetch Market Data & Compute Technical Features
We pull live/historical data (e.g. `AAPL`, `NVDA`, `TSLA`, `SPY`) and compute technical indicators:
- **Trend**: SMA_10, SMA_20, SMA_50, EMA_12, EMA_26
- **Momentum**: RSI_14, MACD, MACD Signal, MACD Histogram
- **Volatility**: Bollinger Bands (Upper, Lower, Width), 20-day Rolling Volatility

In [ ]:
TICKER = "AAPL"
raw_df = fetch_stock_data(symbol=TICKER, period="2y", interval="1d")
df = add_technical_indicators(raw_df)
df.tail()

## 2. Visualize Features & Correlations on Graphs

In [ ]:
# Visualizing feature correlation matrix
corr_features = ["Close", "Open", "High", "Low", "Volume", "SMA_20", "RSI_14", "MACD", "BB_Width", "Volatility_20"]
plot_feature_correlations(df, corr_features, save_path="./outputs/feature_correlations.png")
plot_market_features(df, ticker=TICKER, save_path="./outputs/market_features.png")

# Display inline
from IPython.display import Image, display
display(Image(filename="./outputs/feature_correlations.png"))
display(Image(filename="./outputs/market_features.png"))

## 3. Train & Compare Different Feature Sets
We test 4 different input representations:
1. **Univariate**: Only Close price
2. **Basic Market**: OHLCV (Open, High, Low, Close, Volume)
3. **Market + Trend**: OHLCV + Moving Averages
4. **Full Multi-Feature**: OHLCV + Trend + RSI + MACD + Bollinger Bands + Volatility

In [ ]:
feature_sets = {
    "Univariate (Close Only)": ["Close"],
    "Basic Market (OHLCV)": ["Open", "High", "Low", "Close", "Volume"],
    "Market + Trend (MAs)": ["Open", "High", "Low", "Close", "Volume", "SMA_10", "SMA_20", "EMA_12"],
    "Full Multi-Feature": ["Open", "High", "Low", "Close", "Volume", "SMA_20", "RSI_14", "MACD", "MACD_Signal", "BB_Width", "Volatility_20"]
}

results = run_feature_comparison_experiments(
    df=df,
    feature_sets=feature_sets,
    target_col="Close",
    seq_length=30,
    epochs=30,
    batch_size=32,
    device=device,
    output_dir="./outputs"
)

## 4. Visual Comparison of Model Predictions & Metrics

In [ ]:
display(Image(filename="./outputs/feature_comparison_predictions.png"))
display(Image(filename="./outputs/feature_comparison_loss.png"))
display(Image(filename="./outputs/feature_comparison_metrics.png"))

## 5. Live Prediction from External Data Feed

In [ ]:
# Load best checkpoint and perform live prediction
best_ckpt = "./outputs/lstm_basic_market_(ohlcv).pt"
live_predictor = LiveStockPredictor(checkpoint_path=best_ckpt, device=device)

# Ingest live data and forecast
forecast = live_predictor.predict_from_live_dataframe(raw_df)

print("=" * 50)
print(f"Reference Bar:   {forecast['timestamp']}")
print(f"Current Price:   ${forecast['current_price']:.2f}")
print(f"Predicted Next:  ${forecast['predicted_next_price']:.2f}")
print(f"Predicted Delta: ${forecast['predicted_change']:+.2f} ({forecast['predicted_pct_change']:+.2f}%)")
print(f"Signal:          {forecast['signal']}")
print("=" * 50)